# 🤗 Hugging Face `pipeline()` - Complete Tutorial

## What is the Pipeline Function?

The `pipeline()` function is Hugging Face's **highest-level API** for inference. It abstracts away:
- Model loading and initialization
- Tokenization (preprocessing)
- Model inference
- Post-processing of outputs

### Why Use Pipeline?
| Benefit | Description |
|---------|-------------|
| **Simplicity** | 2-3 lines of code for complex NLP tasks |
| **Automatic Model Selection** | Chooses sensible defaults per task |
| **Device Management** | Handles CPU/GPU placement automatically |
| **Batching** | Efficiently processes multiple inputs |

### Pipeline vs spaCy Comparison
| Feature | HF Pipeline | spaCy |
|---------|-------------|-------|
| **Focus** | Transformer models, SOTA results | Speed, production pipelines |
| **Customization** | Swap models easily | Rule-based + ML hybrid |
| **Memory** | Higher (large models) | Lower footprint |
| **Use Case** | Research, complex NLP | Production, entity extraction |

---

In [ ]:
# Installation (run once if needed)
# !pip install transformers torch datasets accelerate sentencepiece
# !pip install spacy
# !python -m spacy download en_core_web_sm

# Core imports
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

print("✅ Transformers imported successfully!")
print(f"Pipeline function location: {pipeline.__module__}")

---
## 1️⃣ Text Classification (Sentiment Analysis)

### Theory
Text classification assigns labels to text. **Sentiment analysis** is the most common variant, determining if text is positive, negative, or neutral.

### How It Works
```
Input Text → Tokenizer → Transformer Model → Softmax → Label + Score
```

### Models Used
- Default: `distilbert-base-uncased-finetuned-sst-2-english`
- Alternatives: `roberta-base-sentiment`, `cardiffnlp/twitter-roberta-base-sentiment`

### 🔗 spaCy Synergy
spaCy has `TextCategorizer` but HF transformers generally achieve higher accuracy. You can:
- Use spaCy for preprocessing → HF for classification
- Use spaCy's `spacy-transformers` to integrate HF models directly

In [ ]:
# 1. SENTIMENT ANALYSIS / TEXT CLASSIFICATION
classifier = pipeline("sentiment-analysis")

# Single input
result = classifier("I absolutely love this product! Best purchase ever!")
print("Single input result:")
print(f"  Label: {result[0]['label']}, Score: {result[0]['score']:.4f}")

# Multiple inputs (batched)
texts = [
    "This movie was terrible and boring.",
    "The weather is okay today.",
    "I'm so excited about the new features!",
    "The service was disappointing."
]

print("\nBatch processing results:")
results = classifier(texts)
for text, res in zip(texts, results):
    print(f"  '{text[:40]}...' → {res['label']} ({res['score']:.2%})")

In [ ]:
# Using a specific model for multi-class classification
# Twitter sentiment model with 3 classes: negative, neutral, positive
twitter_classifier = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest"
)

tweets = [
    "Just had the worst customer service experience ever 😤",
    "The new update looks interesting",
    "OMG this is amazing!!! 🎉🎉🎉"
]

print("Twitter Sentiment (3-class):")
for tweet in tweets:
    result = twitter_classifier(tweet)
    print(f"  '{tweet[:35]}...' → {result[0]['label']} ({result[0]['score']:.2%})")

---
## 2️⃣ Named Entity Recognition (NER)

### Theory
NER identifies and classifies named entities in text into predefined categories:
- **PER**: Person names
- **ORG**: Organizations  
- **LOC**: Locations
- **MISC**: Miscellaneous entities

### Pipeline Output Structure
```python
{
    'entity': 'B-PER',      # BIO tagging: B=Beginning, I=Inside
    'score': 0.998,         # Confidence
    'index': 5,             # Token position
    'word': 'John',         # The token
    'start': 10,            # Character start position
    'end': 14               # Character end position
}
```

### 🔗 spaCy Synergy
spaCy's NER is **faster** and great for production. Strategy:
- **spaCy**: Fast, rule-augmented NER for high-volume processing
- **HF Pipeline**: When you need SOTA accuracy or domain-specific models
- **Hybrid**: Use spaCy for initial filtering, HF for verification

In [ ]:
# 2. NAMED ENTITY RECOGNITION (NER)
ner_pipeline = pipeline("ner", aggregation_strategy="simple")

text = """
Apple Inc. was founded by Steve Jobs, Steve Wozniak, and Ronald Wayne in Cupertino, California.
The company is now headquartered in Apple Park. Tim Cook has been the CEO since 2011.
"""

print("Named Entity Recognition Results:")
print("-" * 50)
entities = ner_pipeline(text)

for entity in entities:
    print(f"  Entity: '{entity['word']:20}' | Type: {entity['entity_group']:6} | Score: {entity['score']:.2%}")

# Visualize entity positions
print("\n📍 Entity Positions in Text:")
for entity in entities:
    print(f"  [{entity['start']:3}:{entity['end']:3}] {entity['word']} ({entity['entity_group']})")

In [ ]:
# 🔗 SPACY + HUGGING FACE NER COMPARISON
import spacy

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

comparison_text = "Elon Musk announced that Tesla will open a new factory in Berlin, Germany next year."

# spaCy NER
doc = nlp(comparison_text)
print("🔵 spaCy NER Results:")
for ent in doc.ents:
    print(f"  '{ent.text}' → {ent.label_} (chars {ent.start_char}:{ent.end_char})")

# Hugging Face NER
print("\n🟠 Hugging Face NER Results:")
hf_entities = ner_pipeline(comparison_text)
for ent in hf_entities:
    print(f"  '{ent['word']}' → {ent['entity_group']} (score: {ent['score']:.2%})")

print("\n💡 Notice: spaCy uses different labels (PERSON vs PER, GPE vs LOC)")
print("   spaCy is faster, HF often more accurate for complex cases")

---
## 3️⃣ Question Answering (Extractive)

### Theory
**Extractive QA** finds the answer span within a given context. The model:
1. Encodes both question and context
2. Predicts start and end positions of the answer
3. Extracts the text between those positions

### Key Parameters
- `question`: The question to answer
- `context`: The text containing the answer
- `top_k`: Return multiple candidate answers
- `max_answer_len`: Maximum answer length

### Use Cases
- FAQ systems
- Document search
- Customer support automation
- Reading comprehension tasks

In [ ]:
# 3. QUESTION ANSWERING (Extractive)
qa_pipeline = pipeline("question-answering")

context = """
The Transformer architecture was introduced in the paper "Attention Is All You Need" 
published in 2017 by researchers at Google. It revolutionized natural language processing
by replacing recurrent neural networks with self-attention mechanisms. The key innovation
was the ability to process all positions in a sequence simultaneously, making training
much faster. BERT, GPT, and T5 are all based on the Transformer architecture.
"""

questions = [
    "When was the Transformer architecture introduced?",
    "What did Transformers replace?",
    "What models are based on Transformers?",
    "What was the key innovation?"
]

print("📖 Context-based Question Answering")
print("=" * 60)
for q in questions:
    result = qa_pipeline(question=q, context=context)
    print(f"\n❓ Q: {q}")
    print(f"✅ A: {result['answer']} (confidence: {result['score']:.2%})")
    print(f"   Position: chars {result['start']}-{result['end']}")

In [ ]:
# Get multiple candidate answers with top_k
result = qa_pipeline(
    question="What is the Transformer based on?",
    context=context,
    top_k=3  # Get top 3 candidate answers
)

print("🔝 Top-K Answers (multiple candidates):")
for i, ans in enumerate(result, 1):
    print(f"  {i}. '{ans['answer']}' (score: {ans['score']:.4f})")

---
## 4️⃣ Text Summarization

### Theory
Summarization condenses long text while preserving key information.

**Two Types:**
| Type | Description | Models |
|------|-------------|--------|
| **Extractive** | Selects important sentences | TextRank, BERT-ext |
| **Abstractive** | Generates new text | BART, T5, Pegasus |

HF Pipeline uses **abstractive summarization** (generates new sentences).

### Key Parameters
- `max_length`: Maximum summary length
- `min_length`: Minimum summary length  
- `do_sample`: Enable sampling for diversity
- `num_beams`: Beam search width (higher = better quality, slower)

### 🔗 spaCy Synergy
Use spaCy for **extractive summarization** via sentence scoring:
1. spaCy extracts sentences and computes importance
2. HF generates abstractive summary
3. Combine for hybrid approach

In [ ]:
# 4. TEXT SUMMARIZATION
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

article = """
Artificial intelligence has transformed numerous industries over the past decade. 
In healthcare, AI algorithms can now detect diseases from medical images with accuracy 
rivaling human doctors. The financial sector uses AI for fraud detection, algorithmic 
trading, and risk assessment. Manufacturing has embraced AI-powered robotics for 
quality control and predictive maintenance. 

The retail industry leverages AI for personalized recommendations, inventory management, 
and customer service chatbots. Transportation is being revolutionized by autonomous 
vehicles and smart traffic systems. Even creative industries are being impacted, with 
AI generating art, music, and written content.

However, this rapid adoption raises important questions about job displacement, privacy, 
and algorithmic bias. Experts argue that while AI will eliminate some jobs, it will 
create new ones requiring different skills. The key challenge is ensuring that AI 
development benefits society broadly while minimizing potential harms.
"""

print("📄 Original Text Length:", len(article), "characters")
print("\n" + "="*60)

# Generate summary
summary = summarizer(article, max_length=80, min_length=30, do_sample=False)
print("\n📝 Summary:")
print(summary[0]['summary_text'])
print(f"\n📊 Compression: {len(article)} → {len(summary[0]['summary_text'])} chars ({len(summary[0]['summary_text'])/len(article):.1%})")

---
## 5️⃣ Text Generation

### Theory
Text generation produces new text given a prompt. Models predict the next token based on:
- **Autoregressive generation**: Each token depends on previous tokens
- **Sampling strategies**: Control creativity vs coherence

### Generation Parameters
| Parameter | Effect | Typical Values |
|-----------|--------|----------------|
| `max_length` | Maximum tokens to generate | 50-500 |
| `temperature` | Randomness (higher=more creative) | 0.7-1.0 |
| `top_k` | Consider only top k tokens | 50 |
| `top_p` | Nucleus sampling threshold | 0.9-0.95 |
| `do_sample` | Enable probabilistic sampling | True/False |
| `num_return_sequences` | Generate multiple outputs | 1-5 |

### Models
- **GPT-2**: Open-source, various sizes
- **GPT-Neo/J**: Open alternatives to GPT-3
- **BLOOM**: Multilingual open model

In [ ]:
# 5. TEXT GENERATION
generator = pipeline("text-generation", model="gpt2")

prompt = "The future of artificial intelligence is"

# Deterministic generation (greedy)
print("🎯 Deterministic Generation (do_sample=False):")
result = generator(prompt, max_length=50, do_sample=False, pad_token_id=50256)
print(f"  {result[0]['generated_text']}")

# Creative generation with temperature
print("\n🎨 Creative Generation (temperature=0.9):")
result = generator(prompt, max_length=50, do_sample=True, temperature=0.9, 
                   top_p=0.92, pad_token_id=50256)
print(f"  {result[0]['generated_text']}")

# Multiple sequences
print("\n📚 Multiple Generations (num_return_sequences=3):")
results = generator(prompt, max_length=40, do_sample=True, 
                    num_return_sequences=3, temperature=0.8, pad_token_id=50256)
for i, res in enumerate(results, 1):
    print(f"  {i}. {res['generated_text'][:80]}...")

---
## 6️⃣ Translation

### Theory
Neural Machine Translation (NMT) uses encoder-decoder architectures:
1. **Encoder**: Converts source text to hidden representations
2. **Decoder**: Generates target language from representations

### Model Naming Convention
```
Helsinki-NLP/opus-mt-{source}-{target}
```
Example: `Helsinki-NLP/opus-mt-en-de` (English → German)

### Available Language Pairs
HF Hub has 1000+ translation models covering most language pairs.

### 🔗 spaCy Synergy
spaCy doesn't do translation, but complements it:
1. **Pre-translation**: Use spaCy to detect language, clean text
2. **Post-translation**: Use spaCy on translated text for NER, POS tagging
3. **Quality check**: Compare entity counts before/after translation

In [ ]:
# 6. TRANSLATION
# English to French
translator_en_fr = pipeline("translation_en_to_fr", model="Helsinki-NLP/opus-mt-en-fr")

english_texts = [
    "Hello, how are you today?",
    "Machine learning is transforming the world.",
    "The quick brown fox jumps over the lazy dog."
]

print("🇬🇧 → 🇫🇷 English to French Translation:")
print("-" * 50)
for text in english_texts:
    result = translator_en_fr(text)
    print(f"  EN: {text}")
    print(f"  FR: {result[0]['translation_text']}\n")

In [ ]:
# English to German and Spanish
translator_en_de = pipeline("translation_en_to_de", model="Helsinki-NLP/opus-mt-en-de")
translator_en_es = pipeline("translation_en_to_es", model="Helsinki-NLP/opus-mt-en-es")

text = "Artificial intelligence will change how we work and live."

print("🌍 Multi-language Translation:")
print(f"\n  🇬🇧 English:  {text}")
print(f"  🇩🇪 German:   {translator_en_de(text)[0]['translation_text']}")
print(f"  🇪🇸 Spanish:  {translator_en_es(text)[0]['translation_text']}")

---
## 7️⃣ Fill-Mask (Masked Language Modeling)

### Theory
MLM predicts masked tokens in a sequence. This is how BERT was trained!

**Use Cases:**
- Text completion/suggestions
- Data augmentation
- Probing language model knowledge
- Detecting anomalies in text

### How It Works
```
Input:  "The capital of France is [MASK]."
Output: "The capital of France is Paris." (score: 0.95)
```

### 🔗 spaCy Synergy
Combine with spaCy's dependency parsing to create intelligent masks:
- Mask only nouns/verbs for word substitution
- Use syntactic patterns to create meaningful prompts

In [ ]:
# 7. FILL-MASK (Masked Language Modeling)
fill_mask = pipeline("fill-mask", model="bert-base-uncased")

# Basic fill-mask
masked_sentences = [
    "The capital of France is [MASK].",
    "Python is a popular [MASK] language.",
    "The sun rises in the [MASK].",
    "Machine learning requires large amounts of [MASK]."
]

print("🎭 Fill-Mask Predictions:")
print("=" * 60)

for sentence in masked_sentences:
    predictions = fill_mask(sentence)
    print(f"\n📝 '{sentence}'")
    print("   Top predictions:")
    for pred in predictions[:3]:
        print(f"     • {pred['token_str']:15} (score: {pred['score']:.2%})")

In [ ]:
# 🔗 SPACY + FILL-MASK: Intelligent word substitution
# Use spaCy to identify nouns, then mask them for substitution

text = "The scientist discovered a new molecule in the laboratory."
doc = nlp(text)

print("🔗 spaCy-guided Fill-Mask (masking nouns):")
print(f"   Original: {text}\n")

# Find nouns and create masked versions
for token in doc:
    if token.pos_ == "NOUN":
        # Create masked sentence
        masked = text[:token.idx] + "[MASK]" + text[token.idx + len(token.text):]
        predictions = fill_mask(masked)
        
        print(f"   Masked '{token.text}' (NOUN):")
        print(f"     Alternatives: {', '.join([p['token_str'] for p in predictions[:3]])}")

---
## 8️⃣ Zero-Shot Classification

### Theory
Classifies text into categories **without training data** for those categories!

**How It Works:**
1. Converts classification into Natural Language Inference (NLI)
2. For each label, checks: "Does this text entail this label?"
3. Returns probability scores for each label

### Key Parameters
- `candidate_labels`: List of possible categories
- `multi_label`: Allow multiple labels (default: False)
- `hypothesis_template`: Custom template for NLI

### Use Cases
- Rapid prototyping without labeled data
- Dynamic category systems
- Intent classification
- Topic detection

In [ ]:
# 8. ZERO-SHOT CLASSIFICATION
zero_shot = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Single-label classification
text = "The stock market crashed today after the Federal Reserve announced interest rate hikes."
labels = ["politics", "sports", "technology", "finance", "entertainment"]

result = zero_shot(text, candidate_labels=labels)

print("🎯 Zero-Shot Classification (Single Label):")
print(f"   Text: '{text[:60]}...'")
print(f"\n   Scores:")
for label, score in zip(result['labels'], result['scores']):
    bar = "█" * int(score * 20)
    print(f"     {label:15} {bar} {score:.2%}")

In [ ]:
# Multi-label classification (text can belong to multiple categories)
text = "Apple announced a new iPhone with advanced AI features for photography."
labels = ["technology", "business", "photography", "artificial intelligence", "consumer electronics"]

result = zero_shot(text, candidate_labels=labels, multi_label=True)

print("🏷️ Zero-Shot Classification (Multi-Label):")
print(f"   Text: '{text}'")
print(f"\n   Multiple applicable labels:")
for label, score in zip(result['labels'], result['scores']):
    status = "✅" if score > 0.5 else "  "
    bar = "█" * int(score * 20)
    print(f"   {status} {label:25} {bar} {score:.2%}")

In [ ]:
# Custom hypothesis template for intent classification
customer_messages = [
    "I want to cancel my subscription immediately!",
    "How do I reset my password?",
    "Your product is amazing, thank you!",
    "I haven't received my order yet."
]

intent_labels = ["cancel", "technical support", "positive feedback", "order inquiry", "complaint"]

print("💬 Customer Intent Classification:")
print("=" * 60)

for msg in customer_messages:
    result = zero_shot(
        msg, 
        candidate_labels=intent_labels,
        hypothesis_template="This customer message is about {}."
    )
    print(f"\n   '{msg}'")
    print(f"   → Intent: {result['labels'][0]} ({result['scores'][0]:.2%})")

---
## 9️⃣ Text-to-Text Generation (T5/FLAN-T5)

### Theory
Text-to-text models treat ALL NLP tasks as text generation:
- **Translation**: "translate English to German: Hello" → "Hallo"
- **Summarization**: "summarize: [long text]" → "[summary]"
- **Question**: "question: ... context: ..." → "[answer]"

### Key Models
| Model | Size | Best For |
|-------|------|----------|
| `t5-small` | 60M | Quick experiments |
| `t5-base` | 220M | General use |
| `flan-t5-base` | 250M | Instruction following |
| `flan-t5-large` | 780M | Better quality |

### Advantages
- Single model for multiple tasks
- Instruction-tuned versions understand natural language prompts
- Very flexible for custom tasks

In [ ]:
# 9. TEXT-TO-TEXT GENERATION (T5 / FLAN-T5)
text2text = pipeline("text2text-generation", model="google/flan-t5-base")

# Multiple tasks with same model
tasks = [
    "Translate to French: Hello, how are you?",
    "Summarize: Artificial intelligence is a branch of computer science that aims to create intelligent machines. It has become essential in today's industry.",
    "What is the capital of Japan?",
    "Paraphrase: The weather is very nice today.",
    "Sentiment: I absolutely loved this movie, it was fantastic!"
]

print("🔄 Text-to-Text Generation (FLAN-T5):")
print("=" * 60)

for task in tasks:
    result = text2text(task, max_length=100)
    print(f"\n📥 Input:  {task[:60]}...")
    print(f"📤 Output: {result[0]['generated_text']}")

---
## 🔟 Feature Extraction (Embeddings)

### Theory
Feature extraction converts text into dense vector representations (embeddings).

**Applications:**
- Semantic similarity
- Document clustering
- Search/retrieval systems
- Visualization (t-SNE, UMAP)

### Output Shape
```
Input: "Hello world" (2 tokens + special tokens)
Output: [batch_size, sequence_length, hidden_size]
        e.g., [1, 4, 768] for BERT-base
```

### Pooling Strategies
- **CLS token**: Use first token's embedding
- **Mean pooling**: Average all token embeddings
- **Max pooling**: Max across all tokens

### 🔗 spaCy Synergy
Both provide embeddings! Differences:
- **spaCy**: Word2Vec-style, faster, smaller
- **HF**: Contextual embeddings, more nuanced
- Use both for ensemble similarity scoring

In [ ]:
# 10. FEATURE EXTRACTION (Embeddings)
import numpy as np

feature_extractor = pipeline("feature-extraction", model="distilbert-base-uncased")

sentences = [
    "The cat sat on the mat.",
    "A feline rested on the rug.",
    "The stock market crashed today.",
]

print("📊 Feature Extraction (Embeddings):")
print("=" * 60)

embeddings = []
for sent in sentences:
    # Get embeddings (shape: [1, seq_len, hidden_size])
    output = feature_extractor(sent)
    
    # Mean pooling to get sentence embedding
    embedding = np.mean(output[0], axis=0)
    embeddings.append(embedding)
    
    print(f"\n   '{sent}'")
    print(f"   Shape: {np.array(output[0]).shape} → Mean pooled: {embedding.shape}")
    print(f"   Sample values: [{embedding[0]:.4f}, {embedding[1]:.4f}, ... {embedding[-1]:.4f}]")

In [ ]:
# Compute cosine similarity between sentences
from numpy.linalg import norm

def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

print("\n🔗 Semantic Similarity (Cosine):")
print("-" * 40)

# Compare all pairs
for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        sim = cosine_similarity(embeddings[i], embeddings[j])
        print(f"   '{sentences[i][:25]}...' vs '{sentences[j][:25]}...'")
        print(f"   Similarity: {sim:.4f}\n")

print("💡 Notice: 'cat on mat' and 'feline on rug' are semantically similar!")

In [ ]:
# 🔗 SPACY vs HUGGING FACE EMBEDDINGS COMPARISON
print("🔗 spaCy vs Hugging Face Embeddings Comparison:")
print("=" * 60)

test_sentences = [
    "I love programming in Python.",
    "Python programming is my passion.",
    "The snake slithered through the grass."
]

# spaCy embeddings (using en_core_web_sm - note: md/lg models have better vectors)
print("\n🔵 spaCy Embeddings:")
spacy_embeddings = []
for sent in test_sentences:
    doc = nlp(sent)
    spacy_embeddings.append(doc.vector)
    print(f"   '{sent[:35]}...' → shape: {doc.vector.shape}")

# Note: en_core_web_sm has 96-dim vectors
print(f"\n   spaCy vector dimension: {spacy_embeddings[0].shape[0]}")
print(f"   HF vector dimension: {embeddings[0].shape[0]}")

print("\n💡 For better spaCy vectors, use en_core_web_md or en_core_web_lg")

---
## 1️⃣1️⃣ Token Classification (POS Tagging)

### Theory
Token classification assigns labels to individual tokens. Uses include:
- **POS Tagging**: Part-of-speech (noun, verb, adjective...)
- **NER**: Named entity recognition
- **Chunking**: Phrase boundary detection

### 🔗 spaCy Synergy
spaCy excels at POS tagging with:
- Universal POS tags (coarse)
- Fine-grained POS tags
- Dependency labels
- Morphological features

HF models can provide specialized POS for:
- Domain-specific text
- Historical/non-standard text
- Languages where spaCy support is limited

In [ ]:
# 11. TOKEN CLASSIFICATION - spaCy POS Tagging (spaCy is preferred for this)
text = "The quick brown fox jumps over the lazy dog."

doc = nlp(text)

print("🏷️ Part-of-Speech Tagging (spaCy):")
print("=" * 60)
print(f"\n   Text: '{text}'")
print("\n   Token Analysis:")
print(f"   {'Token':<12} {'POS':<8} {'Tag':<8} {'Dependency':<12} {'Head'}")
print("   " + "-" * 52)

for token in doc:
    print(f"   {token.text:<12} {token.pos_:<8} {token.tag_:<8} {token.dep_:<12} {token.head.text}")

# Visualize dependency tree (text version)
print("\n   Noun chunks:", [chunk.text for chunk in doc.noun_chunks])

---
## 1️⃣2️⃣ Conversational AI (Chatbots)

### Theory
Conversational pipelines maintain dialog context and generate appropriate responses.

**Key Components:**
- Conversation history management
- Context encoding
- Response generation

### Models
- `microsoft/DialoGPT-medium`: Open-domain chat
- `facebook/blenderbot-400M-distill`: Social conversation
- `facebook/blenderbot-1B-distill`: Higher quality

### Use Cases
- Customer service bots
- Virtual assistants
- Companionship AI
- Educational tutors

In [ ]:
# 12. CONVERSATIONAL AI
from transformers import Conversation

chatbot = pipeline("conversational", model="facebook/blenderbot-400M-distill")

# Create a conversation
conversation = Conversation("Hello! What do you like to do for fun?")

# Get response
conversation = chatbot(conversation)
print("🤖 Conversational AI (BlenderBot):")
print("=" * 60)
print(f"\n   User:    Hello! What do you like to do for fun?")
print(f"   Bot:     {conversation.generated_responses[-1]}")

# Continue the conversation
conversation.add_user_input("That sounds interesting! Do you read books?")
conversation = chatbot(conversation)
print(f"\n   User:    That sounds interesting! Do you read books?")
print(f"   Bot:     {conversation.generated_responses[-1]}")

# One more turn
conversation.add_user_input("What's your favorite genre?")
conversation = chatbot(conversation)
print(f"\n   User:    What's your favorite genre?")
print(f"   Bot:     {conversation.generated_responses[-1]}")

---
## 1️⃣3️⃣ Table Question Answering

### Theory
TAPAS (Table Parser) answers questions about tabular data without writing SQL.

**How It Works:**
1. Encodes table structure and question
2. Selects relevant cells
3. Performs aggregation operations (sum, average, count)

### Use Cases
- Business intelligence without SQL
- Spreadsheet Q&A
- Financial report analysis
- Data exploration

In [ ]:
# 13. TABLE QUESTION ANSWERING
table_qa = pipeline("table-question-answering", model="google/tapas-base-finetuned-wtq")

# Sample table data
table = {
    "Country": ["USA", "China", "Japan", "Germany", "UK"],
    "Population (M)": ["331", "1402", "125", "83", "67"],
    "GDP (T$)": ["21.4", "14.7", "5.1", "3.9", "2.8"],
    "Continent": ["North America", "Asia", "Asia", "Europe", "Europe"]
}

questions = [
    "What is the population of Japan?",
    "Which country has the highest GDP?",
    "How many countries are in Europe?",
    "What is the total GDP of Asian countries?"
]

print("📊 Table Question Answering (TAPAS):")
print("=" * 60)
print("\n   Table:")
print(f"   {'Country':<15} {'Population':<15} {'GDP':<10} {'Continent'}")
print("   " + "-" * 55)
for i in range(5):
    print(f"   {table['Country'][i]:<15} {table['Population (M)'][i]:<15} {table['GDP (T$)'][i]:<10} {table['Continent'][i]}")

print("\n   Questions & Answers:")
for q in questions:
    result = table_qa(table=table, query=q)
    print(f"\n   ❓ {q}")
    print(f"   ✅ {result['answer']} (aggregator: {result['aggregator']})")

---
## 1️⃣4️⃣ Advanced Pipeline Configuration

### Device Management
```python
# CPU (default)
pipe = pipeline("task")

# Specific GPU
pipe = pipeline("task", device=0)  # GPU 0

# Auto device selection
pipe = pipeline("task", device_map="auto")
```

### Batch Processing
```python
# Process multiple inputs efficiently
pipe(["text1", "text2", "text3"], batch_size=8)
```

### Model Loading Options
- Load from Hub: `model="model-name"`
- Load local: `model="./local/path"`
- Load specific revision: `revision="v1.0"`

In [ ]:
# 14. ADVANCED PIPELINE CONFIGURATION
import torch

# Check available devices
print("⚙️ Pipeline Configuration Options:")
print("=" * 60)

print(f"\n   PyTorch version: {torch.__version__}")
print(f"   CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
print(f"   MPS available (Apple Silicon): {torch.backends.mps.is_available()}")

# Batch processing demonstration
print("\n📦 Batch Processing:")
batch_classifier = pipeline("sentiment-analysis")

texts = [
    "I love this!",
    "This is terrible.",
    "It's okay I guess.",
    "Absolutely fantastic experience!",
    "Never buying this again."
]

# Process in batch
import time
start = time.time()
results = batch_classifier(texts, batch_size=4)
batch_time = time.time() - start

print(f"   Processed {len(texts)} texts in {batch_time:.3f}s")
print(f"   Throughput: {len(texts)/batch_time:.1f} texts/second")

---
## 1️⃣5️⃣ Complete spaCy + Hugging Face Integration Example

### Building a Production NLP Pipeline
This example shows how to combine both libraries for a robust NLP application:

1. **spaCy**: Fast preprocessing, rule-based extraction, linguistic features
2. **Hugging Face**: Deep learning inference for complex tasks
3. **Combined**: Best of both worlds!

In [ ]:
# 15. COMPLETE SPACY + HUGGING FACE INTEGRATION
class HybridNLPPipeline:
    """
    A production-ready NLP pipeline combining spaCy and Hugging Face.
    - spaCy: Fast preprocessing and linguistic analysis
    - HuggingFace: Deep learning inference
    """
    
    def __init__(self):
        # spaCy for preprocessing
        self.nlp = spacy.load("en_core_web_sm")
        
        # HuggingFace for deep learning tasks
        self.sentiment = pipeline("sentiment-analysis")
        self.ner = pipeline("ner", aggregation_strategy="simple")
        self.summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
    
    def analyze(self, text: str) -> dict:
        """Complete analysis of input text."""
        # Step 1: spaCy preprocessing
        doc = self.nlp(text)
        
        # Step 2: Extract linguistic features (spaCy)
        spacy_analysis = {
            "sentences": [sent.text for sent in doc.sents],
            "noun_chunks": [chunk.text for chunk in doc.noun_chunks],
            "entities_spacy": [(ent.text, ent.label_) for ent in doc.ents],
            "num_tokens": len(doc),
            "num_sentences": len(list(doc.sents))
        }
        
        # Step 3: Deep learning inference (HuggingFace)
        hf_analysis = {
            "sentiment": self.sentiment(text[:512])[0],  # Truncate for model
            "entities_hf": self.ner(text[:512])
        }
        
        # Step 4: Summarize if text is long enough
        if len(text) > 200:
            hf_analysis["summary"] = self.summarizer(
                text, max_length=60, min_length=20, do_sample=False
            )[0]["summary_text"]
        
        return {
            "spacy": spacy_analysis,
            "huggingface": hf_analysis,
            "combined_entity_count": len(spacy_analysis["entities_spacy"]) + len(hf_analysis["entities_hf"])
        }

# Initialize the hybrid pipeline
print("🔧 Initializing Hybrid NLP Pipeline...")
hybrid = HybridNLPPipeline()
print("✅ Pipeline ready!")

In [ ]:
# Test the hybrid pipeline
test_article = """
Microsoft announced today that CEO Satya Nadella will lead a new initiative 
to integrate artificial intelligence across all of the company's products. 
The announcement was made at the company's headquarters in Redmond, Washington.
Industry analysts predict this move will significantly impact the technology sector,
with competitors like Google and Amazon expected to respond with their own AI strategies.
The initiative is expected to create thousands of new jobs and drive innovation
in cloud computing, productivity software, and gaming platforms.
"""

print("🔬 Hybrid Pipeline Analysis:")
print("=" * 70)

result = hybrid.analyze(test_article)

print("\n📊 spaCy Results (Fast & Linguistic):")
print(f"   Tokens: {result['spacy']['num_tokens']}")
print(f"   Sentences: {result['spacy']['num_sentences']}")
print(f"   Noun chunks: {result['spacy']['noun_chunks'][:5]}...")
print(f"   Entities: {result['spacy']['entities_spacy']}")

print("\n🤖 Hugging Face Results (Deep Learning):")
print(f"   Sentiment: {result['huggingface']['sentiment']['label']} ({result['huggingface']['sentiment']['score']:.2%})")
print(f"   HF Entities: {[(e['word'], e['entity_group']) for e in result['huggingface']['entities_hf']]}")
print(f"\n📝 Summary: {result['huggingface'].get('summary', 'N/A')}")

print(f"\n📈 Total entities found: {result['combined_entity_count']}")

---
## 📚 Quick Reference: All Pipeline Tasks

| Task | Pipeline Name | Default Model | Use Case |
|------|---------------|---------------|----------|
| Sentiment Analysis | `sentiment-analysis` | distilbert-sst2 | Product reviews, social media |
| Text Classification | `text-classification` | distilbert-sst2 | Topic detection, spam filtering |
| NER | `ner` | bert-base-NER | Information extraction |
| Question Answering | `question-answering` | distilbert-squad | FAQ systems, document search |
| Summarization | `summarization` | bart-large-cnn | News, documents |
| Translation | `translation_xx_to_yy` | opus-mt-* | Localization, global apps |
| Text Generation | `text-generation` | gpt2 | Creative writing, completion |
| Fill-Mask | `fill-mask` | bert-base-uncased | Word prediction, probing |
| Zero-Shot | `zero-shot-classification` | bart-large-mnli | Dynamic classification |
| Text2Text | `text2text-generation` | t5/flan-t5 | Multi-task |
| Feature Extraction | `feature-extraction` | bert/distilbert | Embeddings, similarity |
| Conversation | `conversational` | blenderbot | Chatbots |
| Table QA | `table-question-answering` | tapas | Data analysis |

---
## 🎯 Best Practices Summary

### When to Use HuggingFace Pipeline
✅ Need state-of-the-art accuracy  
✅ Working with complex language understanding  
✅ Research and experimentation  
✅ Zero-shot or few-shot scenarios  

### When to Use spaCy
✅ High-throughput processing  
✅ Rule-based + ML hybrid systems  
✅ Linguistic feature extraction  
✅ Production pipelines with strict latency requirements  

### When to Combine Both
✅ spaCy for preprocessing → HF for inference  
✅ spaCy for fast filtering → HF for precision tasks  
✅ Ensemble approaches for better accuracy  

---
*Created: February 2026 | Transformers library tutorial with spaCy integration*

---
# 🏷️ PART 2: Zero-Shot Topic Tagging — Deep Dive

## Strategies for Zero-Shot Topic Tagging

Zero-shot classification is powerful for topic tagging because you don't need labeled training data. Here are the main strategies:

### Strategy 1: Natural Language Inference (NLI)
The most common approach. Converts classification into entailment:
- **Model**: `facebook/bart-large-mnli`, `roberta-large-mnli`
- **How**: "This text is about {topic}" → entailment score

### Strategy 2: Semantic Similarity (Embeddings)
Compare document embedding to topic embeddings:
- **Models**: `sentence-transformers/all-MiniLM-L6-v2`, `all-mpnet-base-v2`
- **How**: Cosine similarity between text and topic descriptions

### Strategy 3: Prompt-Based (LLMs)
Use instruction-tuned models with explicit prompts:
- **Models**: `google/flan-t5-xl`, `mistralai/Mistral-7B-Instruct`
- **How**: "Classify this text into topics: [list]. Text: {text}"

### Strategy 4: SetFit (Few-Shot → Zero-Shot Transfer)
Train on similar domain, transfer to new labels:
- **Library**: `setfit`
- **How**: Contrastive learning with minimal examples

| Strategy | Pros | Cons | Best For |
|----------|------|------|----------|
| NLI | No training, fast setup | Limited to model's understanding | Quick prototypes |
| Embeddings | Very fast inference | Needs good topic descriptions | Large-scale tagging |
| LLM Prompts | Most flexible | Expensive, slower | Complex taxonomies |
| SetFit | Best accuracy | Needs some examples | Production systems |

In [ ]:
# STRATEGY 1: NLI-Based Zero-Shot (Already covered, but with topic tagging focus)
from transformers import pipeline

zero_shot_nli = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Multi-topic document tagging
document = """
The new quantum computing chip developed by IBM researchers has shown promising results 
in drug discovery applications. The technology could revolutionize how pharmaceutical 
companies develop new medications, potentially reducing the time from years to months.
Scientists are particularly excited about applications in protein folding simulations.
"""

# Hierarchical topic taxonomy
topics = [
    "technology", "healthcare", "business", "science", 
    "quantum computing", "pharmaceuticals", "artificial intelligence",
    "research", "innovation"
]

print("🏷️ Strategy 1: NLI-Based Topic Tagging")
print("=" * 60)

# Multi-label: document can have multiple topics
result = zero_shot_nli(document, candidate_labels=topics, multi_label=True)

print(f"\n📄 Document: '{document[:80]}...'\n")
print("Topics (threshold > 0.3):")
for label, score in zip(result['labels'], result['scores']):
    if score > 0.3:
        bar = "█" * int(score * 30)
        print(f"   ✓ {label:25} {bar} {score:.1%}")

In [ ]:
# STRATEGY 2: Embedding-Based Zero-Shot (Sentence Transformers)
# This is MUCH faster for large-scale tagging

# Install if needed: pip install sentence-transformers
from sentence_transformers import SentenceTransformer, util
import torch

# Load a fast, efficient model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Define topics with descriptions (richer descriptions = better results)
topic_definitions = {
    "technology": "computers, software, hardware, digital innovation, tech industry",
    "healthcare": "medicine, hospitals, doctors, patients, medical treatment, health",
    "finance": "money, banking, stocks, investments, economy, financial markets",
    "science": "research, experiments, discoveries, scientific method, laboratory",
    "sports": "athletics, games, competitions, teams, players, tournaments",
    "politics": "government, elections, policies, politicians, legislation"
}

documents = [
    "The Federal Reserve announced new interest rate policies affecting mortgage markets.",
    "Scientists discovered a new species of deep-sea fish near hydrothermal vents.",
    "The basketball team won the championship after an overtime thriller.",
    "New smartphone features include advanced AI-powered camera systems."
]

print("🏷️ Strategy 2: Embedding-Based Topic Tagging")
print("=" * 60)

# Pre-compute topic embeddings (do this once, cache for production)
topic_names = list(topic_definitions.keys())
topic_texts = list(topic_definitions.values())
topic_embeddings = embedder.encode(topic_texts, convert_to_tensor=True)

# Tag documents
for doc in documents:
    doc_embedding = embedder.encode(doc, convert_to_tensor=True)
    
    # Compute cosine similarity
    similarities = util.cos_sim(doc_embedding, topic_embeddings)[0]
    
    # Get top topics
    top_indices = torch.argsort(similarities, descending=True)[:3]
    
    print(f"\n📄 '{doc[:50]}...'")
    print("   Top topics:")
    for idx in top_indices:
        topic = topic_names[idx]
        score = similarities[idx].item()
        if score > 0.2:  # threshold
            print(f"     • {topic}: {score:.2%}")

In [ ]:
# STRATEGY 3: Prompt-Based with Instruction-Tuned Models
# Using FLAN-T5 for explicit topic classification

text2text = pipeline("text2text-generation", model="google/flan-t5-base")

document = """
Apple's new M3 chip delivers unprecedented performance for machine learning workloads,
making MacBooks competitive with dedicated GPU systems for AI development tasks.
"""

# Explicit prompt engineering for topic tagging
prompt = f"""Classify the following text into one or more of these topics: 
technology, business, artificial intelligence, hardware, software, finance.

Text: {document}

Topics:"""

print("🏷️ Strategy 3: Prompt-Based Topic Tagging (FLAN-T5)")
print("=" * 60)

result = text2text(prompt, max_length=50)
print(f"\n📄 Document: '{document[:60]}...'")
print(f"🏷️ Assigned Topics: {result[0]['generated_text']}")

# More structured prompt
structured_prompt = f"""Task: Multi-label topic classification
Topics to choose from: [technology, business, AI, hardware, consumer electronics]
Text: "{document[:200]}"
Output format: topic1, topic2, topic3
Classification:"""

result2 = text2text(structured_prompt, max_length=30)
print(f"\n🔧 With structured prompt: {result2[0]['generated_text']}")

### 🔗 spaCy Zero-Shot Alternative: spacy-llm

spaCy now has `spacy-llm` for zero-shot classification using LLMs:

```python
# pip install spacy-llm
import spacy

# Configure with OpenAI or local models
nlp = spacy.blank("en")
nlp.add_pipe("llm_textcat", config={
    "model": {"@llm_models": "spacy.GPT-3-5.v1"},
    "task": {"@llm_tasks": "spacy.TextCat.v2", "labels": ["tech", "finance", "sports"]}
})
```

**When to use spacy-llm vs HuggingFace:**
- **spacy-llm**: Integrates into existing spaCy pipelines, production-ready
- **HuggingFace**: More model variety, offline-capable, research-friendly

---
# 🎓 PART 3: Training Your Own Topic Classification Model

## When to Train vs Zero-Shot?

| Scenario | Recommendation |
|----------|----------------|
| < 100 labeled examples | Zero-shot or few-shot |
| 100-1000 examples | SetFit or fine-tuning small models |
| 1000+ examples | Full fine-tuning |
| Domain-specific jargon | Always fine-tune |
| Labels change frequently | Zero-shot |

## Training Routes Overview

```
┌─────────────────────────────────────────────────────────────┐
│                    TRAINING APPROACHES                       │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  FEW-SHOT (10-100 examples)                                 │
│  ├── SetFit (sentence-transformers + classification head)   │
│  └── Pattern Exploiting Training (PET)                      │
│                                                              │
│  FINE-TUNING (100+ examples)                                │
│  ├── 🤗 Transformers Trainer (recommended)                  │
│  ├── PyTorch Lightning                                       │
│  └── Keras/TensorFlow                                        │
│                                                              │
│  TRADITIONAL ML (baseline)                                   │
│  ├── spaCy TextCategorizer                                   │
│  ├── scikit-learn + TF-IDF                                   │
│  └── FastText                                                │
│                                                              │
└─────────────────────────────────────────────────────────────┘
```

---
## Route 1: SetFit (Few-Shot Learning) — Best for Limited Data

**SetFit** = Sentence Transformers + Contrastive Learning + Small Classification Head

### Why SetFit?
- Works with **8-64 examples per class**
- No prompts needed
- Fast training (minutes, not hours)
- Production-ready performance

```bash
pip install setfit
```

In [ ]:
# ROUTE 1: SETFIT - Few-Shot Text Classification
# pip install setfit datasets

from setfit import SetFitModel, Trainer, TrainingArguments
from datasets import Dataset

# Simulate few-shot training data (8 examples per class)
train_data = {
    "text": [
        # Technology
        "The new processor uses advanced 3nm manufacturing technology",
        "Software updates fix critical security vulnerabilities",
        "Cloud computing services experienced an outage",
        "The smartphone features a revolutionary display",
        # Finance
        "Stock markets rallied after positive earnings reports",
        "The central bank raised interest rates by 0.25%",
        "Cryptocurrency prices surged following ETF approval",
        "Mortgage rates hit their lowest point this year",
        # Sports
        "The team won the championship in overtime",
        "Star player signed a record-breaking contract",
        "Olympics committee announced new host city",
        "Tennis rankings updated after Grand Slam",
    ],
    "label": [0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2]  # 0=tech, 1=finance, 2=sports
}

train_dataset = Dataset.from_dict(train_data)

print("🎓 Route 1: SetFit Few-Shot Training")
print("=" * 60)
print(f"Training examples: {len(train_data['text'])} ({len(train_data['text'])//3} per class)")

# Load pre-trained SetFit model
model = SetFitModel.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2",
    labels=["technology", "finance", "sports"]
)

# Training arguments
args = TrainingArguments(
    batch_size=4,
    num_epochs=1,  # SetFit needs very few epochs
    evaluation_strategy="no",
    save_strategy="no",
)

# Create trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
)

print("\n⏳ Training SetFit model (this is fast!)...")
trainer.train()
print("✅ Training complete!")

In [ ]:
# Test the trained SetFit model
test_texts = [
    "Apple announced a new MacBook with M4 chip",
    "Federal Reserve hints at rate cuts next quarter",
    "Manchester United wins Europa League final",
    "SpaceX successfully launched 50 satellites"
]

print("\n🧪 Testing SetFit Model:")
print("-" * 50)

predictions = model.predict(test_texts)
label_names = ["technology", "finance", "sports"]

for text, pred in zip(test_texts, predictions):
    print(f"   '{text[:45]}...'")
    print(f"   → Predicted: {label_names[pred]}\n")

# Get probabilities
probs = model.predict_proba(test_texts)
print("📊 Confidence scores for first example:")
for label, prob in zip(label_names, probs[0]):
    print(f"   {label}: {prob:.2%}")

---
## Route 2: HuggingFace Trainer (Full Fine-Tuning) — The Standard Approach

When you have **500+ labeled examples**, full fine-tuning gives the best results.

### The Training Pipeline
```
1. Prepare Dataset (HF datasets format)
2. Load Pre-trained Model + Tokenizer
3. Configure TrainingArguments
4. Create Trainer
5. Train!
6. Evaluate & Save
```

### Key Libraries
```bash
pip install transformers datasets evaluate accelerate
```

In [ ]:
# ROUTE 2: HUGGINGFACE TRAINER - Full Fine-Tuning
# This is the standard approach for production topic classifiers

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from datasets import Dataset, DatasetDict
import numpy as np
import evaluate

# Step 1: Prepare your dataset
# In real usage, load from CSV/JSON: Dataset.from_csv("data.csv")

train_texts = [
    "The neural network achieved state-of-the-art performance",
    "GPU prices dropped significantly this quarter", 
    "New programming language gains popularity among developers",
    "Bank announces new savings account with higher interest",
    "Stock market volatility increased amid economic uncertainty",
    "Investment funds report record inflows",
    "Soccer team advances to semifinals",
    "Olympic athlete breaks world record",
    "Basketball playoffs begin next week",
] * 20  # Simulating ~180 examples

train_labels = [0, 0, 0, 1, 1, 1, 2, 2, 2] * 20

# Create HuggingFace Dataset
dataset = Dataset.from_dict({
    "text": train_texts,
    "label": train_labels
})

# Split into train/test
dataset = dataset.train_test_split(test_size=0.2, seed=42)

print("🎓 Route 2: HuggingFace Trainer Full Fine-Tuning")
print("=" * 60)
print(f"Training samples: {len(dataset['train'])}")
print(f"Test samples: {len(dataset['test'])}")

# Step 2: Load model and tokenizer
MODEL_NAME = "distilbert-base-uncased"
NUM_LABELS = 3
LABEL_NAMES = ["technology", "finance", "sports"]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label={i: label for i, label in enumerate(LABEL_NAMES)},
    label2id={label: i for i, label in enumerate(LABEL_NAMES)}
)

print(f"\n✅ Loaded model: {MODEL_NAME}")
print(f"   Parameters: {model.num_parameters():,}")

In [ ]:
# Step 3: Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Step 4: Define metrics
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

# Step 5: Configure training
training_args = TrainingArguments(
    output_dir="./topic_classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=10,
    report_to="none",  # Disable wandb/tensorboard for notebook
)

# Step 6: Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

print("⚙️ Training Configuration:")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Batch size: {training_args.per_device_train_batch_size}")
print(f"   Learning rate: {training_args.learning_rate}")
print(f"   Output dir: {training_args.output_dir}")

In [ ]:
# Step 7: Train! (Uncomment to actually train - takes a few minutes)
# trainer.train()

# Step 8: Evaluate
# eval_results = trainer.evaluate()
# print(f"Evaluation accuracy: {eval_results['eval_accuracy']:.2%}")

# Step 9: Save the model
# trainer.save_model("./my_topic_classifier")
# tokenizer.save_pretrained("./my_topic_classifier")

print("\n💡 To train the model, uncomment trainer.train() above")
print("   Training on this data would take ~2-5 minutes on CPU")

# Step 10: Load and use the trained model (after saving)
"""
# Load your trained model
from transformers import pipeline

classifier = pipeline(
    "text-classification", 
    model="./my_topic_classifier",
    tokenizer="./my_topic_classifier"
)

# Use it!
result = classifier("Apple released a new iPhone with AI features")
print(result)  # [{'label': 'technology', 'score': 0.95}]
"""

print("\n📁 After training, your model files will include:")
print("   • config.json (model architecture)")
print("   • model.safetensors (weights)")
print("   • tokenizer.json (tokenizer)")
print("   • training_args.bin (training config)")

---
## Route 3: spaCy TextCategorizer — Production-Ready & Fast

spaCy's `TextCategorizer` is excellent when you need:
- Fast inference
- Integration with other NLP components (NER, POS)
- Smaller model footprint
- Multi-label classification

### Two Architectures Available:
1. **ensemble** (CNN + BoW): Faster, good for most cases
2. **transformer**: Uses HF transformers, highest accuracy

In [ ]:
# ROUTE 3: SPACY TEXTCATEGORIZER
import spacy
from spacy.training import Example
import random

# Create a blank English model
nlp = spacy.blank("en")

# Add the text categorizer with CNN architecture
textcat = nlp.add_pipe("textcat_multilabel")  # or "textcat" for single-label

# Add labels
textcat.add_label("TECHNOLOGY")
textcat.add_label("FINANCE")
textcat.add_label("SPORTS")

# Training data format for spaCy
TRAIN_DATA = [
    ("New AI model achieves breakthrough in language understanding", {"cats": {"TECHNOLOGY": 1.0, "FINANCE": 0.0, "SPORTS": 0.0}}),
    ("The smartphone features an advanced neural processing unit", {"cats": {"TECHNOLOGY": 1.0, "FINANCE": 0.0, "SPORTS": 0.0}}),
    ("Semiconductor shortage affects global chip supply", {"cats": {"TECHNOLOGY": 1.0, "FINANCE": 0.0, "SPORTS": 0.0}}),
    ("Central bank announces interest rate decision", {"cats": {"TECHNOLOGY": 0.0, "FINANCE": 1.0, "SPORTS": 0.0}}),
    ("Stock prices surge on earnings report", {"cats": {"TECHNOLOGY": 0.0, "FINANCE": 1.0, "SPORTS": 0.0}}),
    ("Investors seek safe haven amid market volatility", {"cats": {"TECHNOLOGY": 0.0, "FINANCE": 1.0, "SPORTS": 0.0}}),
    ("Championship finals go into overtime", {"cats": {"TECHNOLOGY": 0.0, "FINANCE": 0.0, "SPORTS": 1.0}}),
    ("Olympic athletes prepare for summer games", {"cats": {"TECHNOLOGY": 0.0, "FINANCE": 0.0, "SPORTS": 1.0}}),
    ("Team signs star player to record contract", {"cats": {"TECHNOLOGY": 0.0, "FINANCE": 0.0, "SPORTS": 1.0}}),
    # Multi-label example (tech company financials)
    ("Apple reports record quarterly revenue from iPhone sales", {"cats": {"TECHNOLOGY": 1.0, "FINANCE": 1.0, "SPORTS": 0.0}}),
]

print("🎓 Route 3: spaCy TextCategorizer Training")
print("=" * 60)
print(f"Training examples: {len(TRAIN_DATA)}")
print(f"Labels: {list(textcat.labels)}")

In [ ]:
# Train the spaCy model
from spacy.training import Example

# Initialize the model
nlp.initialize()

# Training loop
print("\n⏳ Training spaCy TextCategorizer...")
losses_history = []

for epoch in range(10):
    random.shuffle(TRAIN_DATA)
    losses = {}
    
    for text, annotations in TRAIN_DATA:
        doc = nlp.make_doc(text)
        example = Example.from_dict(doc, annotations)
        nlp.update([example], losses=losses)
    
    losses_history.append(losses.get("textcat_multilabel", 0))
    if epoch % 3 == 0:
        print(f"   Epoch {epoch+1:2d} - Loss: {losses.get('textcat_multilabel', 0):.4f}")

print("✅ Training complete!")

# Test the model
test_texts = [
    "NVIDIA announces new graphics card for gaming",
    "Federal Reserve raises rates",
    "World Cup final set for Sunday"
]

print("\n🧪 Testing spaCy Model:")
print("-" * 50)

for text in test_texts:
    doc = nlp(text)
    print(f"\n   '{text}'")
    print("   Predictions:")
    for label, score in sorted(doc.cats.items(), key=lambda x: x[1], reverse=True):
        if score > 0.3:
            print(f"     • {label}: {score:.2%}")

---
## Route 4: scikit-learn + TF-IDF — Classic Baseline

Never underestimate traditional ML! Great for:
- Quick baselines
- When interpretability matters
- Limited compute resources
- Large datasets where transformers are too slow

In [ ]:
# ROUTE 4: SCIKIT-LEARN BASELINE
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

# Training data
texts = [
    "Software update improves battery life", "New processor doubles AI performance",
    "Cloud platform announces price reduction", "Programming framework releases major update",
    "Interest rates expected to rise", "Stock market reaches all-time high",
    "Banking sector reports strong earnings", "Currency exchange rates fluctuate",
    "Team wins championship game", "Athlete breaks world record",
    "Tournament brackets announced", "Coach signs contract extension",
] * 10

labels = ["tech", "tech", "tech", "tech", "finance", "finance", "finance", "finance",
          "sports", "sports", "sports", "sports"] * 10

print("🎓 Route 4: scikit-learn Traditional ML")
print("=" * 60)

# Compare different classifiers
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC()
}

for name, clf in classifiers.items():
    # Create pipeline: TF-IDF → Classifier
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=5000)),
        ('classifier', clf)
    ])
    
    # Cross-validation
    scores = cross_val_score(pipeline, texts, labels, cv=5, scoring='accuracy')
    print(f"\n   {name}:")
    print(f"     Accuracy: {scores.mean():.2%} (+/- {scores.std()*2:.2%})")

In [ ]:
# Train final model and demonstrate usage
final_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=5000)),
    ('classifier', LogisticRegression(max_iter=1000))
])

final_pipeline.fit(texts, labels)

# Test predictions
test_texts = [
    "Apple announces new iPhone with faster chip",
    "Federal Reserve discusses monetary policy",
    "Lakers defeat Celtics in overtime"
]

print("\n🧪 Testing sklearn Model:")
print("-" * 50)

predictions = final_pipeline.predict(test_texts)
probabilities = final_pipeline.predict_proba(test_texts)

for text, pred, probs in zip(test_texts, predictions, probabilities):
    print(f"\n   '{text}'")
    print(f"   → Predicted: {pred}")
    print(f"   Probabilities: {dict(zip(final_pipeline.classes_, [f'{p:.1%}' for p in probs]))}")

# Save model (for production)
import joblib
# joblib.dump(final_pipeline, 'topic_classifier_sklearn.pkl')
# loaded_model = joblib.load('topic_classifier_sklearn.pkl')
print("\n💾 Save with: joblib.dump(pipeline, 'model.pkl')")

---
## 📊 Decision Matrix: Which Approach to Use?

```
                    ┌─────────────────────────────────────────────────────────┐
                    │            TRAINING DATA AVAILABLE?                      │
                    └─────────────────────────────────────────────────────────┘
                                          │
                    ┌─────────────────────┴─────────────────────┐
                    ▼                                           ▼
              NO/MINIMAL                                   YES (100+)
                    │                                           │
        ┌───────────┴───────────┐               ┌───────────────┴───────────────┐
        ▼                       ▼               ▼                               ▼
   ZERO-SHOT              FEW-SHOT         MEDIUM DATA                    LARGE DATA
   (0 examples)           (8-50)           (100-1000)                     (1000+)
        │                     │                 │                              │
        ▼                     ▼                 ▼                              ▼
   ┌─────────┐          ┌─────────┐       ┌──────────┐                 ┌──────────────┐
   │HF Zero- │          │ SetFit  │       │HF Trainer│                 │Full Fine-tune│
   │Shot NLI │          │         │       │ (small)  │                 │   or         │
   │   or    │          │         │       │          │                 │sklearn + TFIDF│
   │Sentence │          │         │       │          │                 │(for speed)    │
   │Transform│          │         │       │          │                 │              │
   └─────────┘          └─────────┘       └──────────┘                 └──────────────┘
```

## 🏆 Recommended Stack by Use Case

| Use Case | Recommended | Why |
|----------|-------------|-----|
| **Rapid Prototyping** | HF Zero-Shot (NLI) | No training, instant results |
| **Production (speed critical)** | spaCy TextCat | Fastest inference, lightweight |
| **Production (accuracy critical)** | SetFit or HF Trainer | Best accuracy with fine-tuning |
| **Limited compute** | sklearn + TF-IDF | Works on any machine |
| **Changing labels** | Zero-Shot or Embeddings | No retraining needed |
| **Domain-specific jargon** | HF Trainer | Learns domain vocabulary |

## 📦 Installation Summary

```bash
# Zero-shot (no training)
pip install transformers sentence-transformers

# Few-shot training
pip install setfit

# Full fine-tuning  
pip install transformers datasets evaluate accelerate

# spaCy training
pip install spacy
python -m spacy download en_core_web_sm

# Traditional ML
pip install scikit-learn
```

---
*End of Topic Classification Deep Dive*